### Tạo dữ liệu và chia batch

In [3]:
import numpy as np

X = np.arange(10, dtype=float).reshape(-1, 1) / 10
y = 3 * X[:, 0] + 2

batch_size = 4
rng = np.random.default_rng(42)
indices = rng.permutation(len(y))

for start in range(0, len(y), batch_size):
    batch_idx = indices[start:start + batch_size]

    X_batch = X[batch_idx]
    y_batch = y[batch_idx]

    print("Chỉ số:", batch_idx)
    print("Shape:", X_batch.shape, y_batch.shape)

Chỉ số: [5 6 0 7]
Shape: (4, 1) (4,)
Chỉ số: [3 2 4 9]
Shape: (4, 1) (4,)
Chỉ số: [1 8]
Shape: (2, 1) (2,)


### MSE và gradient

In [4]:
def mean_squared_error(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)


def linear_gradients(X, y_true, y_pred):
    error = y_pred - y_true
    dw = 2 * X.T @ error / len(y_true)
    db = 2 * error.mean()
    return dw, db

### train bằng mini-batch

In [5]:
class MiniBatchLinearRegression:
    def __init__(
        self,
        learning_rate=0.1,
        n_epochs=1000,
        batch_size=4,
        random_state=42,
    ):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.random_state = random_state

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0.0
        self.loss_history = []
        self.n_updates = 0

        rng = np.random.default_rng(self.random_state)

        for _ in range(self.n_epochs):
            indices = rng.permutation(len(y))

            for start in range(0, len(y), self.batch_size):
                batch_idx = indices[start:start + self.batch_size]
                X_batch = X[batch_idx]
                y_batch = y[batch_idx]

                y_pred = self.predict(X_batch)
                dw, db = linear_gradients(X_batch, y_batch, y_pred)

                self.w -= self.learning_rate * dw
                self.b -= self.learning_rate * db
                self.n_updates += 1

            epoch_loss = mean_squared_error(y, self.predict(X))
            self.loss_history.append(epoch_loss)

        return self

    def predict(self, X):
        return X @ self.w + self.b

### Train và kiểm tra

In [6]:
model = MiniBatchLinearRegression(
    learning_rate=0.1,
    n_epochs=1000,
    batch_size=4,
)

model.fit(X, y)

print("w:", model.w.round(4))
print("b:", round(model.b, 4))
print("MSE cuối:", model.loss_history[-1])
print("Số lần cập nhật:", model.n_updates)

X_new = np.array([[0.25], [0.75]])
print("Dự đoán:", model.predict(X_new).round(4))

w: [3.]
b: 2.0
MSE cuối: 6.113672015462842e-31
Số lần cập nhật: 3000
Dự đoán: [2.75 4.25]
